# InsPLAD Gold v2 Final — Image-only Baseline

최종 freeze된 `gold_v2_final` 246장으로 image-only baseline을 다시 평가합니다.

- Q1 anomaly detection: 246
- Q2 defect classification: 246
- Q3 localization: 42
- 결과 전체는 Drive에 저장
- 핵심 결과는 GitHub `results/latest/gold_v2_baseline/`에 자동 업로드


In [ ]:
# 0) 설정
REPO = 'arnold6444/lvlm_insplad'
REPO_REF = 'main'
QA_FOLDER = 'lvlm_insplad_qa/gold_v2_final'
RESULT_FOLDER = 'lvlm_insplad_results/gold_baseline_v2'
MODELS_TO_RUN = ['qwen3_5_4b','internvl3_5_8b','minicpm_v_4_5','llava_onevision_7b','luxia']
CHECKPOINT_EVERY = 5
SUP_ZIP_NAME = 'supervised_fault_classification.zip'
EXPECTED_SUP_ZIP_SIZE = 874_753_167


In [ ]:
# 1) Drive + repo clone + 환경 준비
import base64, json, os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT = Path('/content/drive/MyDrive')
QA_DIR = DRIVE_ROOT / QA_FOLDER
RESULT_DIR = DRIVE_ROOT / RESULT_FOLDER
RESULT_DIR.mkdir(parents=True, exist_ok=True)
REPO_DIR = Path('/content/lvlm_insplad')
if REPO_DIR.exists(): shutil.rmtree(REPO_DIR)

def run(cmd, cwd=None, env=None):
    cmd=[str(x) for x in cmd]
    print('+', ' '.join(cmd[:8]) + (' ...' if len(cmd)>8 else ''))
    subprocess.run(cmd, cwd=cwd, env=env, check=True)

github_token = userdata.get('GITHUB_TOKEN')
if not github_token: raise RuntimeError('GITHUB_TOKEN이 없습니다.')
os.environ['GITHUB_TOKEN'] = github_token
auth = base64.b64encode(f'x-access-token:{github_token}'.encode()).decode()
git_env=os.environ.copy(); git_env.update({'GIT_CONFIG_COUNT':'1','GIT_CONFIG_KEY_0':'http.extraHeader','GIT_CONFIG_VALUE_0':f'AUTHORIZATION: basic {auth}','GIT_TERMINAL_PROMPT':'0'})
run(['git','clone','--depth','1','--branch',REPO_REF,f'https://github.com/{REPO}.git',REPO_DIR], env=git_env)

for secret in ['HF_TOKEN','LUXIA_API_KEY']:
    try: value=userdata.get(secret)
    except Exception: value=None
    if value: os.environ[secret]=value

run([sys.executable,'-m','pip','install','-q','transformers==5.14.0','accelerate==1.14.0','sentencepiece==0.2.2','pillow','pandas','tqdm'])
print('repo:', subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO_DIR,text=True).strip())
print('QA:', QA_DIR)
print('results:', RESULT_DIR)


In [ ]:
# 2) QA / ZIP / GPU 확인
import torch
if not QA_DIR.is_dir(): raise FileNotFoundError(f'QA folder 없음: {QA_DIR}')
master_files=sorted(QA_DIR.glob('master_qa_*.jsonl'))
loc_files=sorted(QA_DIR.glob('localization_qa_*.jsonl'))
print('master:', master_files[-1] if master_files else None)
print('localization:', loc_files[-1] if loc_files else None)

SUP_ZIP = DRIVE_ROOT / SUP_ZIP_NAME
if not SUP_ZIP.is_file():
    matches=list(DRIVE_ROOT.rglob(SUP_ZIP_NAME))
    if not matches: raise FileNotFoundError(SUP_ZIP_NAME)
    SUP_ZIP=matches[0]
if SUP_ZIP.stat().st_size != EXPECTED_SUP_ZIP_SIZE: raise RuntimeError('supervised ZIP size mismatch')

sys.path.insert(0,str(REPO_DIR/'scripts'))
import run_gold_v2_baselines as runner
rows=runner.load_gold_qa_v2(QA_DIR)
from collections import Counter
print('QA counts:', Counter(r['question_type'] for r in rows))
print('total:', len(rows))
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음')


In [ ]:
# 3) 최종 image-only baseline 실행
run([sys.executable, REPO_DIR/'scripts/run_gold_v2_baselines.py', '--qa-dir', QA_DIR, '--sup-zip', SUP_ZIP, '--out-dir', RESULT_DIR, '--checkpoint-every', str(CHECKPOINT_EVERY), '--models', *MODELS_TO_RUN])


In [ ]:
# 4) 결과 확인 + prediction review 생성
import pandas as pd
leaderboard=pd.read_csv(RESULT_DIR/'leaderboard.csv')
display(leaderboard.sort_values(['anomaly_f1','defect_macro_f1','localization_accuracy'],ascending=False))
if (RESULT_DIR/'metrics_by_asset.csv').is_file(): display(pd.read_csv(RESULT_DIR/'metrics_by_asset.csv'))
print((RESULT_DIR/'run_status.json').read_text(encoding='utf-8'))

qa_rows=runner.base.read_jsonl(RESULT_DIR/'run_qa.jsonl')
qa_by_id={r['id']:r for r in qa_rows}
review=[]
for model_key in MODELS_TO_RUN:
    p=RESULT_DIR/'predictions'/f'{model_key}.jsonl'
    if not p.is_file(): continue
    for pred in runner.base.read_jsonl(p):
        qa=qa_by_id[pred['id']]
        review.append({'model_key':model_key,'id':qa['id'],'core_id':qa['core_id'],'asset_category':qa['asset_category'],'question_type':qa['question_type'],'gt_choice':qa['answer'],'predicted_choice':pred.get('predicted_choice'),'correct':pred.get('predicted_choice')==qa['answer'],'anomaly_gt':qa['anomaly_gt'],'defect_gt':qa['defect_gt'],'raw_output':pred.get('output',''),'error':pred.get('error'),'latency_s':pred.get('latency_s')})
pd.DataFrame(review).to_csv(RESULT_DIR/'prediction_review.csv',index=False)
print('prediction_review rows:',len(review))


In [ ]:
# 5) 핵심 결과 GitHub 자동 publish — 이후 ChatGPT가 바로 확인 가능
run([sys.executable, REPO_DIR/'scripts/publish_results_to_github.py', '--out-dir', RESULT_DIR, '--repo', REPO, '--branch', REPO_REF, '--prefix', 'gold_v2_baseline', '--files', 'leaderboard.csv','metrics_by_asset.csv','run_status.json','run_metadata.json','prediction_review.csv'], env=os.environ.copy())
print('완료: results/latest/gold_v2_baseline/')


## 다음 단계

이 notebook이 끝나면 ChatGPT에 `결과 나왔어 확인해봐`라고 하면 됩니다. 다음은 같은 `gold_v2_final`에서 Reference ablation을 진행합니다.
